# Pengujian Intent & Ekstraksi Entitas (Confusion Matrix + Accuracy/Precision/Recall/F1)

Notebook ini dibuat untuk menguji 1 input contoh (multi-item) pada project BOT.

Output yang ditampilkan:
- Confusion matrix untuk **intent**
- Accuracy/Precision/Recall/F1 untuk **intent** (per kelas + macro/micro/weighted)
- Confusion matrix (2x2) per **field entitas**: TANGGAL, NAMA, BARANG, JUMLAH, SATUAN
- Accuracy/Precision/Recall/F1 per field entitas


In [1]:
import json
import re
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

from nlp.processor import proses_nlp


def _safe_div(a: float, b: float) -> float:
    return a / b if b else 0.0


def _f1(p: float, r: float) -> float:
    return _safe_div(2 * p * r, p + r)


def _norm(v: Any) -> Optional[str]:
    if v is None:
        return None
    if isinstance(v, bool):
        return str(v).lower()
    s = str(v)
    s = " ".join(s.strip().split())
    return s.lower() if s else None


def display_table(rows: List[Dict[str, Any]]):
    try:
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame(rows))
    except Exception:
        print(json.dumps(rows, indent=2, ensure_ascii=False))


def display_matrix(labels: Sequence[str], cm: List[List[int]]):
    try:
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame(cm, index=labels, columns=labels))
    except Exception:
        col_w = max([len(str(x)) for x in labels] + [5])
        header = "".ljust(col_w) + " | " + " ".join(str(l).ljust(col_w) for l in labels)
        sep = "-" * len(header)
        print(header)
        print(sep)
        for i, lab in enumerate(labels):
            row = str(lab).ljust(col_w) + " | " + " ".join(str(cm[i][j]).ljust(col_w) for j in range(len(labels)))
            print(row)


def preprocess_raw_input(raw_text: str) -> str:
    raw_text = raw_text or ""
    lines = [l.rstrip() for l in raw_text.splitlines()]
    lines = [l for l in lines if l.strip()]
    if not lines:
        return ""

    date_line_idx = None
    for i, l in enumerate(lines[:5]):
        if re.fullmatch(r"\d{1,2}-\d{1,2}-\d{2,4}", l.strip()):
            date_line_idx = i
            break

    if date_line_idx is None:
        return "\n".join(lines)

    date_val = lines[date_line_idx].strip()
    name_line_idx = None
    for i, l in enumerate(lines[:10]):
        if re.search(r"\bnama\b\s*:", l, flags=re.IGNORECASE):
            name_line_idx = i
            break

    if name_line_idx is None:
        return "\n".join(lines)

    name_line = lines[name_line_idx].strip()
    header = f"{date_val} {name_line}"
    new_lines = [header]
    for i, l in enumerate(lines):
        if i in (date_line_idx, name_line_idx):
            continue
        new_lines.append(l.strip())
    return "\n".join(new_lines)


def confusion_matrix(y_true: Sequence[str], y_pred: Sequence[str], labels: Optional[Sequence[str]] = None):
    y_true = list(y_true)
    y_pred = list(y_pred)
    n = min(len(y_true), len(y_pred))
    if labels is None:
        labels = sorted(set([str(x) for x in y_true[:n] + y_pred[:n]]))
    labels = list(labels)
    idx = {lab: i for i, lab in enumerate(labels)}
    cm = [[0 for _ in labels] for _ in labels]
    for i in range(n):
        a = str(y_true[i])
        p = str(y_pred[i])
        if a not in idx or p not in idx:
            continue
        cm[idx[a]][idx[p]] += 1
    return labels, cm


def _row_sum(cm, i):
    return sum(cm[i])


def _col_sum(cm, j):
    return sum(cm[i][j] for i in range(len(cm)))


def classification_report_from_cm(labels, cm):
    k = len(labels)
    total = sum(sum(r) for r in cm)
    diag = sum(cm[i][i] for i in range(k))
    accuracy = _safe_div(diag, total)

    per_class = []
    for i, lab in enumerate(labels):
        tp = cm[i][i]
        fp = _col_sum(cm, i) - tp
        fn = _row_sum(cm, i) - tp
        tn = total - (tp + fp + fn)
        support = _row_sum(cm, i)
        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        f1 = _f1(precision, recall)
        per_class.append(
            {
                "label": lab,
                "tp": float(tp),
                "fp": float(fp),
                "fn": float(fn),
                "tn": float(tn),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "support": float(support),
            }
        )

    macro_p = _safe_div(sum(x["precision"] for x in per_class), k)
    macro_r = _safe_div(sum(x["recall"] for x in per_class), k)
    macro_f1 = _safe_div(sum(x["f1"] for x in per_class), k)

    support_total = sum(x["support"] for x in per_class)
    weighted_p = _safe_div(sum(x["precision"] * x["support"] for x in per_class), support_total)
    weighted_r = _safe_div(sum(x["recall"] * x["support"] for x in per_class), support_total)
    weighted_f1 = _safe_div(sum(x["f1"] * x["support"] for x in per_class), support_total)

    tp_micro = sum(cm[i][i] for i in range(k))
    fp_micro = sum(_col_sum(cm, i) - cm[i][i] for i in range(k))
    fn_micro = sum(_row_sum(cm, i) - cm[i][i] for i in range(k))
    micro_p = _safe_div(tp_micro, tp_micro + fp_micro)
    micro_r = _safe_div(tp_micro, tp_micro + fn_micro)
    micro_f1 = _f1(micro_p, micro_r)

    return {
        "accuracy": accuracy,
        "per_class": per_class,
        "macro_avg": {"precision": macro_p, "recall": macro_r, "f1": macro_f1},
        "weighted_avg": {"precision": weighted_p, "recall": weighted_r, "f1": weighted_f1},
        "micro_avg": {"precision": micro_p, "recall": micro_r, "f1": micro_f1},
        "total": float(total),
    }


def field_binary_confusion(true_entities: Sequence[Dict[str, Any]], pred_entities: Sequence[Dict[str, Any]], field: str):
    n = min(len(true_entities), len(pred_entities))
    tp = fp = fn = tn = 0
    for i in range(n):
        t = true_entities[i]
        p = pred_entities[i]
        t_val = _norm(t.get(field))
        p_val = _norm(p.get(field))

        t_has = t_val is not None
        p_has = p_val is not None
        correct = t_has and p_has and (t_val == p_val)

        if t_has:
            if correct:
                tp += 1
            else:
                fn += 1
        else:
            if p_has:
                fp += 1
            else:
                tn += 1

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _f1(precision, recall)
    accuracy = _safe_div(tp + tn, tp + tn + fp + fn)
    labels = ["true", "false"]
    cm = [[tp, fn], [fp, tn]]
    return {
        "field": field,
        "tp": float(tp),
        "fp": float(fp),
        "fn": float(fn),
        "tn": float(tn),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "labels": labels,
        "cm": cm,
    }


In [2]:
raw_input = """19-06-2026 
 
 Nama : Budi budis 
 
 bembeng     40 ctn 
 meses       20 ctn 
 twilo      100 ctn
"""

text_for_nlp = preprocess_raw_input(raw_input)
print("RAW_INPUT")
print(raw_input)
print("\nTEXT_FOR_NLP")
print(text_for_nlp)

pred = proses_nlp(text_for_nlp, already_normalized=True)
print("\nPRED")
print(json.dumps(pred, indent=2, ensure_ascii=False))


RAW_INPUT
19-06-2026 
 
 Nama : Budi budis 
 
 bembeng     40 ctn 
 meses       20 ctn 
 twilo      100 ctn


TEXT_FOR_NLP
19-06-2026 Nama : Budi budis
bembeng     40 ctn
meses       20 ctn
twilo      100 ctn

PRED
[
  {
    "intent": "Catat_Penjualan_Lunas",
    "entitas": {
      "TANGGAL": "19-06-2026",
      "NAMA": "Budi Budis",
      "AKSI": "Tambah Penjualan",
      "BARANG": "Bembeng",
      "JUMLAH": "40 karton",
      "HARGA": null,
      "SATUAN": "karton",
      "TOTAL": null,
      "STATUS": null,
      "NOMINAL_BAYAR": null,
      "METODE_PEMBAYARAN": null,
      "SEMUA": false,
      "KONTEKS_AGREGASI": null,
      "KONDISI": null
    },
    "original_text": "bembeng     40 ctn"
  },
  {
    "intent": "Catat_Penjualan_Lunas",
    "entitas": {
      "TANGGAL": "19-06-2026",
      "NAMA": "Budi Budis",
      "AKSI": "Tambah Penjualan",
      "BARANG": "Meses",
      "JUMLAH": "20 karton",
      "HARGA": null,
      "SATUAN": "karton",
      "TOTAL": null,
      "STATUS": n

In [3]:
ground_truth = [
    {
        "intent": "Catat_Penjualan_Lunas",
        "entitas": {
            "TANGGAL": "19-06-2026",
            "NAMA": "Budi Budis",
            "BARANG": "Bembeng",
            "JUMLAH": "40 karton",
            "SATUAN": "karton",
        },
    },
    {
        "intent": "Catat_Penjualan_Lunas",
        "entitas": {
            "TANGGAL": "19-06-2026",
            "NAMA": "Budi Budis",
            "BARANG": "Meses",
            "JUMLAH": "20 karton",
            "SATUAN": "karton",
        },
    },
    {
        "intent": "Catat_Penjualan_Lunas",
        "entitas": {
            "TANGGAL": "19-06-2026",
            "NAMA": "Budi Budis",
            "BARANG": "Willo",
            "JUMLAH": "100 karton",
            "SATUAN": "karton",
        },
    },
]

y_true_intent = [x.get("intent") for x in ground_truth]
y_pred_intent = [x.get("intent") for x in pred]

print("Y_TRUE_INTENT", y_true_intent)
print("Y_PRED_INTENT", y_pred_intent)


Y_TRUE_INTENT ['Catat_Penjualan_Lunas', 'Catat_Penjualan_Lunas', 'Catat_Penjualan_Lunas']
Y_PRED_INTENT ['Catat_Penjualan_Lunas', 'Catat_Penjualan_Lunas', 'Catat_Penjualan_Lunas']


## Confusion Matrix: Intent + Metrik


In [4]:
labels_intent, cm_intent = confusion_matrix(y_true_intent, y_pred_intent)
print("CONFUSION_MATRIX_INTENT")
display_matrix(labels_intent, cm_intent)

report_intent = classification_report_from_cm(labels_intent, cm_intent)

print("\nPERHITUNGAN_PER_KELAS_INTENT")
display_table(report_intent["per_class"])

print("\nRINGKASAN_INTENT")
summary_intent = [
    {"metric": "accuracy", "value": report_intent["accuracy"]},
    {"metric": "micro_precision", "value": report_intent["micro_avg"]["precision"]},
    {"metric": "micro_recall", "value": report_intent["micro_avg"]["recall"]},
    {"metric": "micro_f1", "value": report_intent["micro_avg"]["f1"]},
    {"metric": "macro_precision", "value": report_intent["macro_avg"]["precision"]},
    {"metric": "macro_recall", "value": report_intent["macro_avg"]["recall"]},
    {"metric": "macro_f1", "value": report_intent["macro_avg"]["f1"]},
    {"metric": "weighted_precision", "value": report_intent["weighted_avg"]["precision"]},
    {"metric": "weighted_recall", "value": report_intent["weighted_avg"]["recall"]},
    {"metric": "weighted_f1", "value": report_intent["weighted_avg"]["f1"]},
    {"metric": "total_samples", "value": report_intent["total"]},
]
display_table(summary_intent)


CONFUSION_MATRIX_INTENT


,Catat_Penjualan_Lunas
Catat_Penjualan_Lunas,3



PERHITUNGAN_PER_KELAS_INTENT


,label,tp,fp,fn,tn,precision,recall,f1,support
0,Catat_Penjualan_Lunas,3.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0



RINGKASAN_INTENT


,metric,value
0,accuracy,1.0
1,micro_precision,1.0
2,micro_recall,1.0
3,micro_f1,1.0
4,macro_precision,1.0
5,macro_recall,1.0
6,macro_f1,1.0
7,weighted_precision,1.0
8,weighted_recall,1.0
9,weighted_f1,1.0


## Confusion Matrix: Ekstraksi Entitas (per Field) + Metrik

Definisi 2x2 per field:
- TP: field ada di ground truth dan **nilai prediksi sama persis**
- FN: field ada di ground truth tapi **hilang / salah nilai**
- FP: field tidak ada di ground truth tapi model memprediksi ada
- TN: field tidak ada di ground truth dan model juga tidak memprediksi


In [5]:
fields = ["TANGGAL", "NAMA", "BARANG", "JUMLAH", "SATUAN"]
true_entities = [x.get("entitas") or {} for x in ground_truth]
pred_entities = [x.get("entitas") or {} for x in pred]

rows = []
for f in fields:
    m = field_binary_confusion(true_entities, pred_entities, f)
    rows.append({k: v for k, v in m.items() if k not in {"labels", "cm"}})

print("METRIK_PER_FIELD")
display_table(rows)

for f in fields:
    m = field_binary_confusion(true_entities, pred_entities, f)
    print(f"\nCONFUSION_MATRIX_FIELD={f}")
    display_matrix(["true", "false"], m["cm"])


METRIK_PER_FIELD


,field,tp,fp,fn,tn,accuracy,precision,recall,f1
0,TANGGAL,3.0,0.0,0.0,0.0,1.000000,1.0,1.000000,1.0
1,NAMA,2.0,0.0,1.0,0.0,0.666667,1.0,0.666667,0.8
2,BARANG,3.0,0.0,0.0,0.0,1.000000,1.0,1.000000,1.0
3,JUMLAH,3.0,0.0,0.0,0.0,1.000000,1.0,1.000000,1.0
4,SATUAN,3.0,0.0,0.0,0.0,1.000000,1.0,1.000000,1.0



CONFUSION_MATRIX_FIELD=TANGGAL


,true,false
true,3,0
false,0,0



CONFUSION_MATRIX_FIELD=NAMA


...

,true,false
true,3,0
false,0,0



CONFUSION_MATRIX_FIELD=JUMLAH


,true,false
true,3,0
false,0,0



CONFUSION_MATRIX_FIELD=SATUAN


,true,false
true,3,0
false,0,0


## Detail Per Item (Entitas)


In [6]:
detail_rows = []
n = min(len(true_entities), len(pred_entities))
for i in range(n):
    r: Dict[str, Any] = {"item": i + 1}
    for f in fields:
        r[f"true_{f}"] = true_entities[i].get(f)
        r[f"pred_{f}"] = pred_entities[i].get(f)
        r[f"match_{f}"] = _norm(true_entities[i].get(f)) == _norm(pred_entities[i].get(f))
    detail_rows.append(r)

display_table(detail_rows)


,item,true_TANGGAL,pred_TANGGAL,match_TANGGAL,true_NAMA,pred_NAMA,match_NAMA,true_BARANG,pred_BARANG,match_BARANG,true_JUMLAH,pred_JUMLAH,match_JUMLAH,true_SATUAN,pred_SATUAN,match_SATUAN
0,1,19-06-2026,19-06-2026,True,Budi Budis,Budi Budis,True,Bembeng,Bembeng,True,40 karton,40 karton,True,karton,karton,True
1,2,19-06-2026,19-06-2026,True,Budi Budis,Budi Budis,True,Meses,Meses,True,20 karton,20 karton,True,karton,karton,True
2,3,19-06-2026,19-06-2026,True,Budi Budis,Twilo,False,Willo,Willo,True,100 karton,100 karton,True,karton,karton,True
